# Edición de Imágenes - Ejercicio complementario

En este Notebook vamos a afianzar algunos de los conceptos que hemos trabajado en clase, realizando diversos ejercicios sobre **edición de imágenes**

- Edición de fondos: `product-background`
- Ver los beneficios de la automatización y escala que nos permite la IA Generativa


# Objetivo
El objetivo de este ejercicio es ser capaz de generar múltiples variaciones d

En este caso, trabajamos para una tienda de productos de viaje y vamos a lanzar una nueva maleta. Queremos ampliar nuestro contenido audiovisual mediante la edición de fondos y la expansión a diferentes formatos

### Imagen inicial
https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png

### Resultado esperado
Para la imagen inicial, vamos a obtener .

Las siguientes variaciones sirven como ejemplo/inspiración ¡sientete libre para experimentar con las variaciones que quieras!:
- Suelo de la sala de espera de un aeropuerto
- Encima de una cama king-size en una habitación
- En el suelo de un salón, delante de un tronco de Brasil
- ...

### Instalar el SDK de Python de Vertex AI

Instala las dependencias necesarias para realizar llamadas programáticas a los modelos de Google

In [ ]:
%pip install --upgrade --quiet google-genai

### Autentica tu entorno de cuaderno (solo Colab)

Si estás ejecutando este cuaderno en Google Colab, ejecuta la siguiente celda para autenticar tu entorno.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Importa las librerias necesarias


In [ ]:
from google import genai
from google.genai.types import (
    EditImageConfig,
    GenerateImagesConfig,
    Image,
    MaskReferenceConfig,
    MaskReferenceImage,
    RawReferenceImage,
)

### Inicializar Vertex AI en nuestro Google Cloud Project

In [ ]:
# Importar Vertex AI
import vertexai

# Definir la info del proyecto
PROJECT_ID = ""  # @param {type:"string"}
# Vamos a utilizar esta localización por defecto
LOCATION = "us-central1"

# Inicializar el módulo
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

Vamos a verificar que todo está en orden y qué modo estamos utilizando



In [ ]:
if not client.vertexai:
    print("Usando Gemini Developer API.")
elif client._api_client.project:
    print(
        f"Usando Vertex AI en el proyecto: {client._api_client.project} en la localización: {client._api_client.location}"
    )
elif client._api_client.api_key:
    print(
        f"Usando Vertex AI en modo express con API key: {client._api_client.api_key[:5]}...{client._api_client.api_key[-5:]}"
    )

###  Funciones auxiliares
Las siguientes funciones auxiliares te van a permitir realizar el ejercicio. No tienes que modificar nada en su código, solamente ejecutar este bloque

In [ ]:
import io
import IPython.display

import matplotlib.pyplot as plt
import math
import urllib
import typing

from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps

def display_image(
    image,
    max_width: int = 700,
    max_height: int = 400,
) -> None:
    pil_image = typing.cast(PIL_Image.Image, image._pil_image)
    if pil_image.mode != "RGB":
        # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
        pil_image = pil_image.convert("RGB")
    image_width, image_height = pil_image.size
    if max_width < image_width or max_height < image_height:
        # Resize to display a smaller notebook image
        pil_image = pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
    IPython.display.display(pil_image)

def display_images_in_grid(images, cols=4):
    n = len(images)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(15, 5 * rows))

    for i, img_obj in enumerate(images):
        plt.subplot(rows, cols, i + 1)

        # --- CORRECCIÓN AQUÍ ---
        # Convertimos los bytes de la respuesta en una imagen de PIL
        image_data = PIL_Image.open(io.BytesIO(img_obj.image.image_bytes))

        plt.imshow(image_data)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

## Ejercicio 1 - Creación de prompts
Vamos a crear varios prompts para editar el fondo de la imagen de nuestra maleta mediante un bucle de iteración en python.

In [ ]:
# TODO: Añade tus prompts para generar variaciones de fondos
# Para poder iterar programáticamente, una lista es una buena estructura para almacenarlos
background_prompts = [
]

## Ejercicio 2 - Editar backgrounds a escala automáticamente

Vamos a ver la imagen original de la cual vamos a generar diferentes variaciones

In [ ]:
product_image = Image(
    gcs_uri="gs://cloud-samples-data/generative-ai/image/suitcase.png"
)
raw_ref_image = RawReferenceImage(reference_image=product_image, reference_id=0)

mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=None,
    config=MaskReferenceConfig(mask_mode="MASK_MODE_BACKGROUND"),
)

product_image_show = PIL_Image.open(
      urllib.request.urlopen(
          "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png"
      )
  )

# Display generated images
fig, axis = plt.subplots(1, 2, figsize=(12, 6))
axis[0].imshow(product_image_show)
axis[0].set_title("Original")
for ax in axis:
    ax.axis("off")
plt.show()

Completa el código iterativo para generar variaciones para cada prompt.

https://www.w3schools.com/python/python_for_loops.asp

In [ ]:
# TODO: Itera sobre los prompts y genera las ediciones de fondos de forma automatizada
for ... in ...:
  response = client.models.edit_image(
    model="",  # Utiliza el modelo de edición de imagenes correspondiente
    prompt="",  # Utiliza un prompt dinámico (variable iterada en el bucle for)
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # Elige el modo de edición correspondiente
        number_of_images=4,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
  )

  # Mostrar las imagenes generadas junto con su prompt
  print("")  # TODO: Añade la variable prompt al print para ver en pantalla el prompt y las imagenes generadas
  display_images_in_grid(response.generated_images)
  print("-"*50)